# IS453 Group Assignment — G1-7 — Medical Procedure Loan
## Credit scorecard: full pipeline in one notebook

From the two raw course files to the final scorecard, cutoff and pricing, using the methods from the Week 9–11 labs.
The write-up `IS453_Group_7_Combined_Notebook_Writeup.md` explains every step and every change from the first version
(`credit_scorecard_medical.py`).

| Part | What it does | Brief component |
|---|---|---|
| 1 | Segment filtering, checkpoint numbers | 2. Data selection |
| 2 | Bureau data cleaning, flattening and left join | 2. Data preparation |
| 3 | Modelling table, correlated variables, train/test split | 2. Data preparation |
| 4 | Fine classing, IV screening, coarse classing | 3. Screening and grouping |
| 5 | Logistic regression, scorecard, AUC/Gini/KS, PCC measures | 4. Logistic regression and performance |
| 6 | Model tuning comparison | 5. Model tuning |
| 7 | Cutoff, expected defaults, profit, break-even interest rate | 6. Cutoff, interest rate, profitability |
| 8 | Three example applicants scored point by point | 7. Scorecard walkthrough |

**Files needed** (in the same folder as this notebook, the same way as `hmeq_data.csv` in the labs):
* `IS453 Group Assignment - Application Data.csv`
* `IS453 Group Assignment - Bureau Data.csv`

**Files this notebook saves** (in the same folder): `g7_iv_fine_classing.csv` (IV of every variable)
`g7_scorecard_points.csv` (the final scorecard) and
`g7_final_bins_and_points.csv` (the final model's bins with counts, bad rates, WOE and points).

The whole notebook takes a few minutes to run. Binning about 40 variables with `sc.woebin` is the slowest step.

## 0. Setup
Same packages as the Week 11 lab. After the two `pip install` lines finish, restart the kernel
and run the cell again, as the lab says. If you see an error about numpy "dtype size changed",
change the first line to `!pip install pandas==2.1.1 "numpy<2"` and restart the kernel again.

In [ ]:
!pip install pandas==2.1.1
!pip install scorecardpy==0.1.9.7
# after the installs finish: restart the kernel, then run this cell again

import warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='scorecardpy')

import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import linear_model, metrics
import scorecardpy as sc
import pprint

print('pandas', pd.__version__, '| scorecardpy', sc.__version__)

## Part 1. Segment filtering

**Segment definition** (the group repo's `credit_scorecard_medical.py`):

| # | Filter | Reason |
|---|---|---|
| 1 | `NAME_CONTRACT_TYPE == 'Cash loans'` | a procedure is a one-off cost repaid in fixed instalments, not a revolving credit line |
| 2 | `50,000 <= AMT_CREDIT <= 500,000` | the typical cost range of a medical procedure in the data's currency: leaves out very small purchases and large property or car-sized loans |
| 3 | `25 <= age <= 65` (`-DAYS_BIRTH / 365.25`) | independent adults with elective or restorative medical needs; the upper limit keeps the loan from running into advanced age |
| 4 | `NAME_INCOME_TYPE` in Working, Commercial associate, State servant, Pensioner | a regular income to repay from (excludes students and the unemployed) |
| 5 | `AMT_ANNUITY` not missing and `AMT_INCOME_TOTAL > 0` | the instalment and income must be known to judge affordability |

`ORGANIZATION_TYPE == 'Medicine'` is **not** used: it describes where the applicant works, not why they borrow.
TARGET is only used to count bads. It played no part in choosing the filters.

In [ ]:
app = pd.read_csv('IS453 Group Assignment - Application Data.csv', low_memory=False)
assert app['SK_ID_CURR'].is_unique, 'each row should be one application'
app['AGE_YEARS'] = -app['DAYS_BIRTH'] / 365.25

In [ ]:
f1 = app['NAME_CONTRACT_TYPE'] == 'Cash loans'
f2 = app['AMT_CREDIT'].between(50000, 500000)
f3 = app['AGE_YEARS'].between(25, 65)
f4 = app['NAME_INCOME_TYPE'].isin(['Working', 'Commercial associate', 'State servant', 'Pensioner'])
f5 = app['AMT_ANNUITY'].notna() & (app['AMT_INCOME_TOTAL'] > 0)

steps = [('Original data', pd.Series(True, index=app.index)),
         ('1. Cash loans', f1),
         ('2. + loan amount 50k-500k', f1 & f2),
         ('3. + age 25-65', f1 & f2 & f3),
         ('4. + income type', f1 & f2 & f3 & f4),
         ('5. + annuity known, income > 0', f1 & f2 & f3 & f4 & f5)]

funnel = pd.DataFrame([dict(step=name, rows=int(m.sum()), bads=int(app.loc[m, 'TARGET'].sum()),
                            bad_rate=app.loc[m, 'TARGET'].mean()) for name, m in steps]).set_index('step')
funnel['share_of_original'] = funnel['rows'] / len(app)
funnel.style.format({'rows': '{:,}', 'bads': '{:,}', 'bad_rate': '{:.2%}', 'share_of_original': '{:.1%}'})

In [ ]:
seg = app[f1 & f2 & f3 & f4 & f5].copy()
base_rate, seg_rate = app['TARGET'].mean(), seg['TARGET'].mean()
print(f"Segment: {len(seg):,} rows, {int(seg['TARGET'].sum()):,} bads, bad rate {seg_rate:.2%}")
print(f"Original: {len(app):,} rows, bad rate {base_rate:.2%}  ->  segment is {seg_rate/base_rate:.2f}x the original bad rate")

**Why the bad rate is higher than the original.** Almost all of the change comes from the loan-amount filter. Among cash
loans, the largest loans (above 500k) are the safest, and mid-sized loans of 250k–500k are the riskiest. The segment keeps
the mid-sized loans and removes the large ones. The age filter also keeps 25–29 year-olds, the riskiest age group.

In [ ]:
cash = app[f1]
credit_band = pd.cut(cash['AMT_CREDIT'], [0, 50000, 250000, 500000, 750000, 1000000, np.inf])
cash.groupby(credit_band, observed=True)['TARGET'].agg(rows='size', bad_rate='mean').style.format({'rows': '{:,}', 'bad_rate': '{:.2%}'})

In [ ]:
age_band = pd.cut(seg['AGE_YEARS'], [25, 30, 35, 45, 55, 65.01], right=False)
seg.groupby(age_band, observed=True)['TARGET'].agg(rows='size', bad_rate='mean').style.format({'rows': '{:,}', 'bad_rate': '{:.2%}'})

**Does it look like a procedure-loan segment?** Compared with all cash loans, the segment has smaller loans and shorter
terms, which fits one-off procedure costs rather than cars or property.

In [ ]:
compare = pd.DataFrame({
    'All cash loans': [cash['AMT_CREDIT'].median(), (cash['AMT_CREDIT'] / cash['AMT_ANNUITY']).median(), cash['AMT_INCOME_TOTAL'].median()],
    'Segment':        [seg['AMT_CREDIT'].median(),  (seg['AMT_CREDIT'] / seg['AMT_ANNUITY']).median(),   seg['AMT_INCOME_TOTAL'].median()]},
    index=['median loan amount', 'median term (months)', 'median income'])
compare.round(1)

## Part 2. Bureau data: why it cannot be joined directly
The application data has **one row per applicant**. The bureau data has **one row per credit the applicant holds at
other lenders**. A direct left join (Option 1 in the data-merging module) would repeat each applicant once per credit and
create false extra "bad" rows. Spreading credits into columns (Option 2) would give many mostly empty columns. So we use
**Option 3**: summarise each applicant's credits into a set of features, then join one row per applicant.

In [ ]:
bureau = pd.read_csv('IS453 Group Assignment - Bureau Data.csv', low_memory=False)
per_applicant = bureau.groupby('SK_ID_CURR').size()
print(f'bureau rows: {len(bureau):,} | applicants: {bureau["SK_ID_CURR"].nunique():,} | '
      f'credits per applicant: median {per_applicant.median():.0f}, max {per_applicant.max()}')

### 2.1 Data quality checks
Run before summarising, because a bad value becomes invisible once it is added into a total.

In [ ]:
print('-- issues to fix --')
print(f'negative AMT_CREDIT_SUM_DEBT  : {(bureau.AMT_CREDIT_SUM_DEBT < 0).sum():,}  (a debt cannot be negative)')
print(f'negative AMT_CREDIT_SUM_LIMIT : {(bureau.AMT_CREDIT_SUM_LIMIT < 0).sum():,}  (a limit cannot be negative)')
print(f'DAYS_CREDIT_UPDATE > 0        : {(bureau.DAYS_CREDIT_UPDATE > 0).sum():,}  (updated after the application = future information)')
print(f'DAYS_CREDIT > 0               : {(bureau.DAYS_CREDIT > 0).sum():,}')
print()
print('-- not an issue --')
print(f'DAYS_CREDIT_ENDDATE > 0       : {(bureau.DAYS_CREDIT_ENDDATE > 0).sum():,}  (the credit is still running)')
print()
print(bureau['CREDIT_CURRENCY'].value_counts().to_string())

### 2.2 Clean
* Drop the credits updated **after** the application date. The bank would not have had that information when deciding (leakage).
* Set negative debts and limits to 0.
* 99.9% of credits are in currency 1 and no exchange rates are given, so other currencies are **flagged**, not converted.
  Their amounts are still added in. Their effect is negligible.

In [ ]:
b = bureau[bureau['DAYS_CREDIT_UPDATE'] <= 0].copy()
b['AMT_CREDIT_SUM_DEBT']  = b['AMT_CREDIT_SUM_DEBT'].clip(lower=0)
b['AMT_CREDIT_SUM_LIMIT'] = b['AMT_CREDIT_SUM_LIMIT'].clip(lower=0)
b['IS_FOREIGN_CCY'] = (b['CREDIT_CURRENCY'] != 'currency 1').astype(int)
print(f'{len(bureau):,} -> {len(b):,} rows ({len(bureau) - len(b)} dropped as post-application updates)')

### 2.3 Yes/no flags for each credit
Each flag is 1 or 0 for each credit, so adding them up per applicant gives a count.
`DAYS_CREDIT` is negative (days before the application), so "opened in the last 12 months" means `DAYS_CREDIT >= -365`.

In [ ]:
b['F_ACTIVE']       = (b['CREDIT_ACTIVE'] == 'Active').astype(int)
b['F_CLOSED']       = (b['CREDIT_ACTIVE'] == 'Closed').astype(int)
b['F_BAD_DEBT']     = b['CREDIT_ACTIVE'].isin(['Bad debt', 'Sold']).astype(int)
b['F_OVERDUE_NOW']  = (b['CREDIT_DAY_OVERDUE'] > 0).astype(int)
b['F_EVER_OVERDUE'] = ((b['CREDIT_DAY_OVERDUE'] > 0) | (b['AMT_CREDIT_MAX_OVERDUE'].fillna(0) > 0)).astype(int)
b['F_PROLONGED']    = (b['CNT_CREDIT_PROLONG'] > 0).astype(int)
b['F_CONSUMER']     = (b['CREDIT_TYPE'] == 'Consumer credit').astype(int)
b['F_CARD']         = (b['CREDIT_TYPE'] == 'Credit card').astype(int)
b['F_MICROLOAN']    = (b['CREDIT_TYPE'] == 'Microloan').astype(int)
b['F_SECURED']      = b['CREDIT_TYPE'].isin(['Car loan', 'Mortgage', 'Real estate loan']).astype(int)
b['F_OPENED_12M']   = (b['DAYS_CREDIT'] >= -365).astype(int)
b['F_OPENED_24M']   = (b['DAYS_CREDIT'] >= -730).astype(int)
# amounts on active credits only, so paid-off credits do not count as current debt
b['ACT_DEBT']   = b['AMT_CREDIT_SUM_DEBT'].where(b['F_ACTIVE'] == 1)
b['ACT_CREDIT'] = b['AMT_CREDIT_SUM'].where(b['F_ACTIVE'] == 1)

### 2.4 Summarise to one row per applicant
Only the segment's applicants are summarised. `groupby('SK_ID_CURR').agg(...)` works like a pivot table: one row per
applicant, with each new column built as `NEW_NAME = ('column', 'how')`.

| Family | Features |
|---|---|
| Account counts | total, active, closed, sold / bad debt |
| Delinquency | overdue now, ever overdue, prolonged, worst days and amount overdue |
| Exposure | total credit and debt, active credit and debt, largest and average credit |
| Credit mix | consumer, card, microloan, secured, number of credit types, foreign currency |
| History and recency | first and latest credit, opened in the last 12 and 24 months |

In [ ]:
b_seg = b[b['SK_ID_CURR'].isin(seg['SK_ID_CURR'])]

agg = b_seg.groupby('SK_ID_CURR').agg(
    BUR_N_CREDITS        = ('SK_ID_BUREAU', 'count'),
    BUR_N_ACTIVE         = ('F_ACTIVE', 'sum'),
    BUR_N_CLOSED         = ('F_CLOSED', 'sum'),
    BUR_N_BAD_DEBT       = ('F_BAD_DEBT', 'sum'),
    BUR_N_OVERDUE_NOW    = ('F_OVERDUE_NOW', 'sum'),
    BUR_N_EVER_OVERDUE   = ('F_EVER_OVERDUE', 'sum'),
    BUR_N_PROLONGED      = ('F_PROLONGED', 'sum'),
    BUR_MAX_DAYS_OVERDUE = ('CREDIT_DAY_OVERDUE', 'max'),
    BUR_MAX_AMT_OVERDUE  = ('AMT_CREDIT_MAX_OVERDUE', 'max'),
    BUR_SUM_OVERDUE_NOW  = ('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    BUR_SUM_CREDIT       = ('AMT_CREDIT_SUM', 'sum'),
    BUR_SUM_DEBT         = ('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUR_MAX_CREDIT       = ('AMT_CREDIT_SUM', 'max'),
    BUR_MEAN_CREDIT      = ('AMT_CREDIT_SUM', 'mean'),
    BUR_SUM_ACT_DEBT     = ('ACT_DEBT', 'sum'),
    BUR_SUM_ACT_CREDIT   = ('ACT_CREDIT', 'sum'),
    BUR_N_CONSUMER       = ('F_CONSUMER', 'sum'),
    BUR_N_CARD           = ('F_CARD', 'sum'),
    BUR_N_MICROLOAN      = ('F_MICROLOAN', 'sum'),
    BUR_N_SECURED        = ('F_SECURED', 'sum'),
    BUR_N_CREDIT_TYPES   = ('CREDIT_TYPE', 'nunique'),
    BUR_N_FOREIGN_CCY    = ('IS_FOREIGN_CCY', 'sum'),
    BUR_DAYS_FIRST       = ('DAYS_CREDIT', 'min'),
    BUR_DAYS_LAST        = ('DAYS_CREDIT', 'max'),
    BUR_N_OPENED_12M     = ('F_OPENED_12M', 'sum'),
    BUR_N_OPENED_24M     = ('F_OPENED_24M', 'sum'),
).reset_index()

# ratios: a 0 denominator gives a blank (NaN), never 0, so the binning gives it a Missing bin
agg['BUR_UTILISATION']       = agg['BUR_SUM_ACT_DEBT'] / agg['BUR_SUM_ACT_CREDIT'].replace(0, np.nan)
agg['BUR_DEBT_TO_CREDIT']    = agg['BUR_SUM_DEBT'] / agg['BUR_SUM_CREDIT'].replace(0, np.nan)
agg['BUR_SHARE_ACTIVE']      = agg['BUR_N_ACTIVE'] / agg['BUR_N_CREDITS']
agg['BUR_SHARE_OVERDUE']     = agg['BUR_N_EVER_OVERDUE'] / agg['BUR_N_CREDITS']
agg['BUR_HISTORY_YEARS']     = -agg['BUR_DAYS_FIRST'] / 365.25
agg['BUR_MONTHS_SINCE_LAST'] = -agg['BUR_DAYS_LAST'] / 30.44
agg['BUR_EVER_OVERDUE_FLAG'] = (agg['BUR_N_EVER_OVERDUE'] > 0).astype(int)

assert agg['SK_ID_CURR'].is_unique, 'must be one row per applicant'
print(f'{len(b_seg):,} bureau rows for the segment -> {len(agg):,} applicants x {agg.shape[1] - 1} features')
agg.head()

### 2.5 Left join and missing values
A **left join** keeps every applicant in the segment, including the ones with no bureau record. An inner join would
drop them, and they are a riskier group.

For applicants with no bureau record:

| Feature type | Filled with | Why |
|---|---|---|
| Counts (`BUR_N_...`) and totals (`BUR_SUM_...`), the ever-overdue flag | **0** | no credits means zero accounts and zero debt |
| Ratios, averages, maximums, dates | **left blank** | e.g. a utilisation of 0 would claim they borrow and never use it. A blank gets its own Missing bin later |

In [ ]:
zero_fill = [c for c in agg.columns if c.startswith(('BUR_N_', 'BUR_SUM_'))] + ['BUR_EVER_OVERDUE_FLAG']
keep_blank = [c for c in agg.columns if c not in zero_fill + ['SK_ID_CURR']]

model_df = seg.merge(agg, on='SK_ID_CURR', how='left', validate='one_to_one')
model_df['HAS_BUREAU'] = model_df['BUR_N_CREDITS'].notna().astype(int)
model_df[zero_fill] = model_df[zero_fill].fillna(0)

# checks: same applicants, nobody duplicated, bads unchanged
assert len(model_df) == len(seg), 'row count changed during the merge'
assert model_df['SK_ID_CURR'].is_unique, 'an applicant appears twice'
assert model_df['TARGET'].sum() == seg['TARGET'].sum(), 'number of bads changed'
print(f'merged: {model_df.shape[0]:,} rows x {model_df.shape[1]} columns - all checks passed')
print(f'filled with 0 ({len(zero_fill)}): {", ".join(zero_fill)}')
print(f'left blank ({len(keep_blank)}): {", ".join(keep_blank)}')

### 2.6 Applicants with no bureau record (thin files)

In [ ]:
thin = model_df.groupby('HAS_BUREAU')['TARGET'].agg(rows='size', bads='sum', bad_rate='mean')
thin['share'] = thin['rows'] / len(model_df)
thin.index = thin.index.map({0: 'No bureau record', 1: 'Has bureau record'})
thin.style.format({'rows': '{:,}', 'bads': '{:,}', 'bad_rate': '{:.2%}', 'share': '{:.1%}'})

### 2.7 Checkpoint summary

In [ ]:
print('=' * 70)
print('G1-7 Medical procedure loan - segment and merge summary')
print('=' * 70)
print('filters: cash loans, loan amount 50k-500k, age 25-65, income type, annuity known and income > 0')
print(f'segment: {len(seg):,} rows ({len(seg)/len(app):.1%} of the original {len(app):,})')
print(f'bad rate: {seg_rate:.2%} vs {base_rate:.2%} in the original data')
print(f'bureau: {len(bureau):,} credit rows -> {len(agg):,} applicants with a record, {agg.shape[1]-1} features each')
print(f'merge: left join on SK_ID_CURR, {model_df.shape[0]:,} rows x {model_df.shape[1]} columns, one row per applicant')
print(f"thin files: {(model_df.HAS_BUREAU == 0).sum():,} ({(model_df.HAS_BUREAU == 0).mean():.1%}), "
      f"bad rate {model_df.loc[model_df.HAS_BUREAU == 0, 'TARGET'].mean():.2%} "
      f"vs {model_df.loc[model_df.HAS_BUREAU == 1, 'TARGET'].mean():.2%} with a record")

## Part 3. Modelling table and train/test split
### 3.1 Build the modelling table
Start from the merged table built in Part 2 (application and bureau columns, one row per applicant).
Then, as in the Week 9 lab DIY Q2, **create ratio variables** that combine two fields into something more meaningful:

| New variable | Meaning |
|---|---|
| `CREDIT_TO_INCOME`, `ANNUITY_TO_INCOME` | loan size and monthly instalment relative to income (affordability) |
| `CREDIT_TO_GOODS` | loan amount relative to the procedure price (amount financed on top of the price) |
| `TERM_MONTHS` | loan term, `AMT_CREDIT / AMT_ANNUITY` |
| `DEBT_TO_INCOME` | (existing active bureau debt + this loan) / income |
| `BUR_ANY_MICROLOAN`, `BUR_ANY_OVERDUE_NOW`, `BUR_ANY_BAD_DEBT` | yes/no versions of rare but severe bureau events |
| `ORGANIZATION_TYPE` | sub-types such as "Business Entity Type 1/2/3" grouped into their sector (58 values → 35) |

Day counts are turned into years so the bins are readable. `DAYS_EMPLOYED = 365243` is a placeholder used for pensioners
and the unemployed, so it is set to missing and flagged.

In [ ]:
# merged segment table from Part 2: ID, TARGET, bureau features, then the original application columns
flat_cols = ['SK_ID_CURR', 'TARGET'] + [c for c in agg.columns if c != 'SK_ID_CURR'] + ['HAS_BUREAU']
app_cols  = [c for c in app.columns if c not in ('SK_ID_CURR', 'TARGET', 'AGE_YEARS')]
df = model_df[flat_cols + app_cols].copy()
assert len(df) == len(seg) and df['SK_ID_CURR'].is_unique

# readable time variables
df['AGE_YEARS']          = -df['DAYS_BIRTH'] / 365.25
df['EMP_ANOMALY']        = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['YEARS_EMPLOYED']     = -df['DAYS_EMPLOYED'].where(df['DAYS_EMPLOYED'] != 365243) / 365.25
df['YEARS_REGISTRATION'] = -df['DAYS_REGISTRATION'] / 365.25
df['YEARS_ID_PUBLISH']   = -df['DAYS_ID_PUBLISH'] / 365.25
df['YEARS_PHONE_CHANGE'] = -df['DAYS_LAST_PHONE_CHANGE'] / 365.25
df = df.drop(columns=['DAYS_BIRTH','DAYS_EMPLOYED','DAYS_REGISTRATION','DAYS_ID_PUBLISH','DAYS_LAST_PHONE_CHANGE'])

# ratio variables (Week 9 DIY Q2 idea)
df['CREDIT_TO_INCOME']  = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_TO_INCOME'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['CREDIT_TO_GOODS']   = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']
df['TERM_MONTHS']       = df['AMT_CREDIT'] / df['AMT_ANNUITY']
df['DEBT_TO_INCOME']    = (df['BUR_SUM_ACT_DEBT'] + df['AMT_CREDIT']) / df['AMT_INCOME_TOTAL']
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
df['BUR_ANY_MICROLOAN']   = (df['BUR_N_MICROLOAN'] > 0).astype(int)
df['BUR_ANY_OVERDUE_NOW'] = (df['BUR_N_OVERDUE_NOW'] > 0).astype(int)
df['BUR_ANY_BAD_DEBT']    = (df['BUR_N_BAD_DEBT'] > 0).astype(int)

# ORGANIZATION_TYPE has 58 values ('Business Entity Type 1/2/3', 'Trade: type 1..7', ...). sc.woebin stops and asks for
# confirmation when a text variable has 50+ values, so group the sub-types into their parent sector (35 values)
df['ORGANIZATION_TYPE'] = df['ORGANIZATION_TYPE'].astype(str).str.replace(r':? [Tt]ype \d+$', '', regex=True)

# not used for modelling
df = df.drop(columns=['CODE_GENDER',                     # protected attribute: not used for credit decisions
                      'BUR_DAYS_FIRST','BUR_DAYS_LAST'])  # same information as BUR_HISTORY_YEARS / BUR_MONTHS_SINCE_LAST
const = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
df = df.drop(columns=const)
print('constant columns dropped:', const)

# keep loan amount and term for the profit calculation in Part 7 (AMT_CREDIT may be dropped as a model input below)
loan_info = df[['SK_ID_CURR', 'AMT_CREDIT', 'TERM_MONTHS']].copy()
print(df.shape)

### 3.2 Data preparation (Week 9 lab)
**Highly correlated variables.** As in Week 9 Ex Q2: for every pair of numeric variables with correlation above 0.8 or
below -0.8, drop the one with the weaker correlation to TARGET. Many pairs are the building statistics, which come in
`_AVG`, `_MODE` and `_MEDI` versions of the same measurement.

In [ ]:
num = df.drop(columns=['SK_ID_CURR']).select_dtypes('number')
cor = num.corr()
pairs = cor.where(np.triu(np.ones(cor.shape), 1).astype(bool)).stack()      # each pair once
high_corr = pairs[pairs.abs() > 0.8].sort_values(key=abs, ascending=False)

corr_dropped = set()
for (a, b), r in high_corr.items():
    if 'TARGET' in (a, b) or a in corr_dropped or b in corr_dropped:
        continue
    corr_dropped.add(a if abs(cor.loc[a, 'TARGET']) < abs(cor.loc[b, 'TARGET']) else b)

print(f'{len(high_corr)} pairs with |corr| > 0.8 -> dropped {len(corr_dropped)} variables')
print(high_corr.head(10).round(3).to_string())
df = df.drop(columns=sorted(corr_dropped))

**Train-test split** (Week 11 lab): `sc.split_df` keeps 70% for training and 30% for testing, with the same bad rate in both,
using a fixed random seed. All binning and model choices use the **training set only**. The test set shows how the
scorecard performs on applicants it has not seen.

In [ ]:
model_data = df.drop(columns=['SK_ID_CURR'])
train, test = sc.split_df(model_data, y='TARGET', ratio=.7).values()
print('train', train.shape, f"bad rate {train['TARGET'].mean():.2%}")
print('test ', test.shape,  f"bad rate {test['TARGET'].mean():.2%}")

## Part 4. Screening and grouping of variables
### 4.1 Screening: fine classing and IV (Week 10 lab)
Fine classing with `pd.qcut` into **20 equal-frequency bins** (variables with 20 or fewer distinct values use one bin per
value), then the lab's `woe_iv` function: **WOE = ln(% non-events / % events)** and **IV = Σ (% non-events − % events) × WOE**.

| IV | Predictive power |
|---|---|
| < 0.02 | unpredictive, dropped |
| 0.02 – 0.1 | weak |
| 0.1 – 0.3 | medium |
| > 0.3 | strong |

A fine bin with no bads gives an infinite WOE. Those bins are left out of the IV total here, and coarse classing merges them anyway.

In [ ]:
# Week 10 lab function (TARGET in place of BAD)
def woe_iv(data):
    working_data = data.copy()
    working_data['Bin_Range'] = working_data['Bin_Range'].astype('object')
    working_data['Bin_Range'] = working_data['Bin_Range'].fillna('Missing')

    variable_data = pd.DataFrame()
    variable_data['Bin_Range'] = working_data.groupby(by='Bin_Range', as_index=False).count()['Bin_Range']
    variable_data['Count'] = working_data.groupby(by='Bin_Range', as_index=False).count()['TARGET']
    variable_data['Events'] = working_data.groupby(by='Bin_Range', as_index=False).sum()['TARGET']
    variable_data['Non_Events'] = variable_data['Count'] - variable_data['Events']
    variable_data['%_of_Events'] = variable_data['Events'] / sum(variable_data['Events'])
    variable_data['%_of_Non_Events'] = variable_data['Non_Events'] / sum(variable_data['Non_Events'])
    variable_data['WOE'] = np.log(variable_data['%_of_Non_Events'] / variable_data['%_of_Events'])
    variable_data['IV'] = (variable_data['%_of_Non_Events'] - variable_data['%_of_Events']) * variable_data['WOE']
    variable_data['total_IV'] = variable_data['IV'].sum()
    return variable_data

def fine_class(data, variable, bin_count=20):
    d = data[[variable, 'TARGET']].copy()
    if pd.api.types.is_numeric_dtype(d[variable]) and d[variable].nunique() > bin_count:
        d['Bin_Range'] = pd.qcut(d[variable], q=bin_count, duplicates='drop')
    else:
        d['Bin_Range'] = d[variable]
    return d

def iv_strength(v):
    return 'unpredictive' if v < 0.02 else 'weak' if v < 0.1 else 'medium' if v < 0.3 else 'strong'

rows = []
for variable in train.columns.drop('TARGET'):
    t = woe_iv(fine_class(train, variable))
    finite = np.isfinite(t['IV'])
    rows.append(dict(variable=variable, IV_fine=t.loc[finite, 'IV'].sum(), n_bins=len(t), zero_bad_bins=int((~finite).sum())))
iv_fine = pd.DataFrame(rows).sort_values('IV_fine', ascending=False).reset_index(drop=True)
iv_fine['strength'] = iv_fine['IV_fine'].map(iv_strength)
iv_fine.to_csv('g7_iv_fine_classing.csv', index=False)

screened = iv_fine.loc[iv_fine['IV_fine'] >= 0.02, 'variable'].tolist()
print(iv_fine['strength'].value_counts().to_string())
print(f'\n{len(screened)} variables kept (IV >= 0.02)')
iv_fine.head(len(screened)).round(4)

**WOE plot** (Week 10 lab `woe_iv_plot`) for the top variables: line = WOE per fine bin, bars = % of applicants.

In [ ]:
def woe_iv_plot(data, variable_name):
    variable_data = woe_iv(data)
    print('IV for', variable_name, 'with', variable_data.shape[0]-1, 'numeric bins:', round(variable_data.loc[0,'total_IV'],4))
    fig, ax1 = plt.subplots(figsize=(20,6))
    if variable_data['Bin_Range'].isin(['Missing']).sum() > 0:
        sns.lineplot(data = variable_data['WOE'].iloc[0:-1], marker='o', ax=ax1)
    else:
        sns.lineplot(data = variable_data['WOE'], marker='o', ax=ax1)
    plt.xticks(rotation=45)
    if variable_data['Bin_Range'].isin(['Missing']).sum() > 0:
        last_point = len(variable_data['WOE']) - 1
        sns.scatterplot(x=[last_point], y=[variable_data['WOE'][last_point]], color='b', s=30, ax=ax1)
    ax2 = ax1.twinx()
    total_count = variable_data['Count'].sum()
    variable_data['Percent'] = (variable_data['Count'] / total_count) * 100
    sns.barplot(x = variable_data['Bin_Range'].astype(str), y = variable_data['Percent'], alpha=0.5, ax = ax2, color = 'deepskyblue')
    ax2.set_ylabel('Percent of Observations (%)')
    plt.title("WOE plot for " + variable_name)
    plt.show()

for variable in ['YEARS_EMPLOYED', 'TERM_MONTHS']:
    woe_iv_plot(fine_class(train, variable), variable)

### 4.2 Grouping: coarse classing (Week 10 and Week 11 labs)
`sc.woebin` builds the coarse bins automatically. It optimises IV, gives every bin at least 5% of applicants and creates a
missing bin, but **it does not force a monotonic trend** (Week 11 lab note). Following Week 10 Ex Q3, every numeric
variable is checked for a monotonic bad rate, and the ones that break the trend get adjusted `breaks_list` bin edges.

**Note on sign:** scorecardpy defines WOE the other way round from the Week 10 function:
**scorecardpy WOE = ln(% bads / % goods)**, so a *positive* WOE means a *riskier* bin. The IV is the same either way.

In [ ]:
bins_auto = sc.woebin(train[screened + ['TARGET']], y='TARGET')

def is_monotonic(b):
    r = b.loc[b['bin'] != 'missing', 'badprob'].values
    d = np.diff(r)
    return len(r) < 3 or bool((d >= 0).all() or (d <= 0).all())

auto_summary = pd.DataFrame([dict(variable=k, IV_coarse=v['total_iv'].iloc[0], n_bins=len(v), monotonic=is_monotonic(v),
                                  numeric=pd.api.types.is_numeric_dtype(train[k])) for k, v in bins_auto.items()])
auto_summary = auto_summary.sort_values('IV_coarse', ascending=False).reset_index(drop=True)
print('not monotonic (numeric):', auto_summary.query('~monotonic and numeric')['variable'].tolist())
auto_summary.round(4)

In [ ]:
# show the automatic bins of the variables that are not monotonic
for v in auto_summary.query('~monotonic and numeric')['variable']:
    display(bins_auto[v][['variable','bin','count','count_distr','badprob','woe','total_iv']].round(4))

**Adjusted bin edges.** For each non-monotonic numeric variable, adjacent bins whose bad rates cross are merged, so
that the bad rate rises or falls steadily (Week 10 Ex Q3). The resulting edges are given to `sc.woebin` as a `breaks_list`.
The comment next to each one gives the business reading.

In [ ]:
breaks_adj = {
    # merge bins whose bad rates cross, so the trend runs one way (Week 10 Ex Q3)
    'CREDIT_TO_GOODS':              [1.14, 1.25],             # financing more than the procedure price -> riskier
    'AMT_CREDIT':                   [110000, 180000, 280000], # larger loan -> riskier
    'AMT_ANNUITY':                  [13000, 16000],           # larger monthly instalment -> riskier
    'REGION_POPULATION_RELATIVE':   [0.021, 0.032],           # more populated region -> safer
    'YEARS_BEGINEXPLUATATION_MEDI': [0.977, 0.988],           # newer building -> safer
    'BUR_MAX_CREDIT':               [100000, 2450000],        # larger past credits elsewhere -> safer
    # U-shaped: kept as 3 bins, one of which breaks the trend (Week 10 DIY Q1 approach)
    'TERM_MONTHS':                  [10.5, 21.5],             # short (under ~10m) and long (22m+) loans safer than 11-21m
}

bins = sc.woebin(train[screened + ['TARGET']], y='TARGET', breaks_list=breaks_adj)

coarse_summary = pd.DataFrame([dict(variable=k, IV_coarse=v['total_iv'].iloc[0], n_bins=len(v), monotonic=is_monotonic(v))
                               for k, v in bins.items()]).sort_values('IV_coarse', ascending=False).reset_index(drop=True)
coarse_summary['strength'] = coarse_summary['IV_coarse'].map(iv_strength)
print('still not monotonic:', coarse_summary.query('~monotonic')['variable'].tolist())

# second screen: drop variables whose IV fell below 0.02 after grouping
coarse_kept = coarse_summary.loc[coarse_summary['IV_coarse'] >= 0.02, 'variable'].tolist()
print(f'{len(coarse_kept)} variables with coarse IV >= 0.02')
coarse_summary.round(4)

In [ ]:
# adjusted bins for the variables that were changed
for v in breaks_adj:
    if v in bins:
        display(bins[v][['variable','bin','count','count_distr','badprob','woe','total_iv']].round(4))

## Part 5. Logistic regression and model performance
### 5.1 Logistic regression with WOE encoding (Week 11 lab)
`sc.woebin_ply` replaces each value with the WOE of its bin. Before fitting:
* **Correlation between WOE variables:** same 0.8 rule as Week 9, keeping the variable with the higher IV.
* **Variable selection** (in the spirit of Week 9 DIY Q3): forward selection. At each step add the variable that raises the
  training **Gini** the most, and stop at 15 variables or when the gain drops below 0.002. A variable is accepted only if
  **every coefficient stays above 0.1**. With scorecardpy's WOE (positive = riskier), a negative coefficient would make the
  points run against the bins, and a coefficient near 0 means a variable has been made redundant by a similar one added
  later, which leaves it with the same points in every bin.

In [ ]:
train_woe = sc.woebin_ply(train[coarse_kept + ['TARGET']], bins)
test_woe  = sc.woebin_ply(test[coarse_kept + ['TARGET']], bins)
y_train, y_test = train_woe['TARGET'], test_woe['TARGET']

iv_of = coarse_summary.set_index('variable')['IV_coarse']
woe_cols = sorted([c for c in train_woe.columns if c.endswith('_woe')], key=lambda c: -iv_of[c[:-4]])
wcor = train_woe[woe_cols].corr().abs()
candidates = []
for c in woe_cols:                                       # highest IV first
    if all(wcor.loc[c, k] <= 0.8 for k in candidates):
        candidates.append(c)
print(f'{len(woe_cols)} WOE variables -> {len(candidates)} after dropping |corr| > 0.8')

def gini(y, p):
    return 2 * metrics.roc_auc_score(y, p) - 1

def fit(cols):
    lr = linear_model.LogisticRegression(max_iter=1000)
    lr.fit(train_woe[cols], y_train)
    return lr

MIN_COEF = 0.1   # every coefficient must stay clearly positive
selected, path, best_g = [], [], 0.0
while len(selected) < 15:
    best = None
    for c in candidates:
        if c in selected:
            continue
        lr = fit(selected + [c])
        if (lr.coef_[0] < MIN_COEF).any():               # wrong sign, or a variable pushed to ~0 weight -> reject
            continue
        g = gini(y_train, lr.predict_proba(train_woe[selected + [c]])[:, 1])
        if best is None or g > best[1]:
            best = (c, g, lr)
    if best is None or best[1] - best_g < 0.002:
        break
    c, best_g, lr = best
    selected.append(c)
    path.append(dict(step=len(selected), added=c[:-4], gini_train=best_g,
                     gini_test=gini(y_test, lr.predict_proba(test_woe[selected])[:, 1])))
path = pd.DataFrame(path)
path.round(4)

In [ ]:
# final model (same call as the Week 11 lab)
X_train, X_test = train_woe[selected], test_woe[selected]
lr = linear_model.LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
coefs = pd.Series(np.concatenate([lr.intercept_, lr.coef_[0]]), index=np.concatenate([['intercept'], lr.feature_names_in_]))
pd.DataFrame({'coef': coefs, 'IV': [np.nan] + [iv_of[c[:-4]] for c in selected]}).round(4)

### 5.2 Scorecard and model performance (Week 11 lab)
Same scaling as the lab: **600 points = odds of 1:19 (bad:good), i.e. a 5% probability of default**, and **20 points
doubles the odds (PDO = 20)**. `basepoints_eq0=True` spreads the base points across the variables.

In [ ]:
card = sc.scorecard(bins, lr, X_train.columns, points0 = 600, odds0 = 1/19, pdo = 20, basepoints_eq0 = True)
scorecard_table = pd.concat(card.values(), ignore_index=True).query("variable != 'basepoints'")[['variable','bin','points']]
scorecard_table.to_csv('g7_scorecard_points.csv', index=False)

train_score = sc.scorecard_ply(train, card)
test_score  = sc.scorecard_ply(test, card)
print('score range', int(min(train_score.score.min(), test_score.score.min())), '-', int(max(train_score.score.max(), test_score.score.max())))
scorecard_table

In [ ]:
# bins of the final model with their points (the table for the slides)
final_bins = pd.concat([bins[c[:-4]] for c in selected], ignore_index=True)
final_bins = final_bins.merge(scorecard_table, on=['variable', 'bin'], how='left')
final_bins = final_bins[['variable', 'bin', 'count', 'count_distr', 'badprob', 'woe', 'total_iv', 'points']]
final_bins.to_csv('g7_final_bins_and_points.csv', index=False)
final_bins.round(4)

**Ranking performance (AUC, Gini, KS)** on train and test. Similar values on both mean the model is not overfitting.

In [ ]:
def ks_stat(y, score):
    # cumulative shares of bads and goods at or below each score (applicants with the same score move together)
    d = pd.DataFrame({'y': np.asarray(y), 's': np.asarray(score)}).groupby('s')['y'].agg(bads='sum', n='size').cumsum()
    cum_bad = d['bads'] / d['bads'].iloc[-1]
    cum_good = (d['n'] - d['bads']) / (d['n'] - d['bads']).iloc[-1]
    return (cum_bad - cum_good).abs().max()

perf = pd.DataFrame({
    'AUC':  [metrics.roc_auc_score(y_train, -train_score.score), metrics.roc_auc_score(y_test, -test_score.score)],
    'Gini': [gini(y_train, -train_score.score), gini(y_test, -test_score.score)],
    'KS':   [ks_stat(y_train, train_score.score), ks_stat(y_test, test_score.score)]}, index=['train', 'test'])
perf.round(4)

**ROC and K-S curves.** For these curves a *defaulter* is the event the score is trying to catch, so a low score means
"predicted defaulter".
* **ROC curve (left):** working down from the lowest scores, the share of defaulters caught (y-axis) against the share
  of good payers wrongly caught with them (x-axis). The further the curve bows above the diagonal, the better the
  scorecard ranks applicants. The area under it is the AUC (0.5 = random, 1 = perfect).
* **K-S chart (right):** for every score, the share of all defaulters and the share of all good payers scoring at or
  below it. The K-S statistic is the widest gap between the two lines, i.e. the score where the scorecard separates
  goods from bads most strongly.

In [ ]:
fpr_tr, tpr_tr, _ = metrics.roc_curve(y_train, -train_score['score'].values)
fpr_te, tpr_te, _ = metrics.roc_curve(y_test, -test_score['score'].values)

ks_df = pd.DataFrame({'y': y_test.values, 'score': test_score['score'].values}).sort_values('score')
ks_df = ks_df.groupby('score')['y'].agg(bads='sum', n='size').cumsum()
ks_df['cum_bads'] = ks_df['bads'] / ks_df['bads'].iloc[-1]
ks_df['cum_goods'] = (ks_df['n'] - ks_df['bads']) / (ks_df['n'] - ks_df['bads']).iloc[-1]
ks_df['gap'] = ks_df['cum_bads'] - ks_df['cum_goods']
ks_score = ks_df['gap'].idxmax()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(fpr_tr, tpr_tr, lw=2, label=f"Train (AUC = {perf.loc['train', 'AUC']:.3f})")
axes[0].plot(fpr_te, tpr_te, lw=2, label=f"Test (AUC = {perf.loc['test', 'AUC']:.3f})")
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
axes[0].set_xlabel('Share of good payers caught (false positive rate)')
axes[0].set_ylabel('Share of defaulters caught (true positive rate)')
axes[0].set_title('ROC curve'); axes[0].legend(loc='lower right')

axes[1].plot(ks_df.index, ks_df['cum_bads'], lw=2, color='firebrick', label='Defaulters at or below score')
axes[1].plot(ks_df.index, ks_df['cum_goods'], lw=2, color='seagreen', label='Good payers at or below score')
axes[1].axvline(ks_score, ls='--', c='grey', label=f"K-S = {ks_df['gap'].max():.1%} at score {ks_score:.0f}")
axes[1].set_xlabel('Score'); axes[1].set_ylabel('Cumulative share')
axes[1].set_title('K-S chart (test set)'); axes[1].legend(loc='lower right')
plt.tight_layout(); plt.savefig('g7_roc_ks_curves.png', dpi=150); plt.show()
print(f"Test K-S {ks_df['gap'].max():.2%} at score {ks_score:.0f}")

**Score distribution** (Week 11 lab)

In [ ]:
combined_score = pd.concat([train_score, test_score], ignore_index=True)
combined_score.hist(figsize = (7, 4), bins = 60)
plt.tight_layout()
plt.savefig('g7_score_distribution.png', dpi=150)

**PCC measures at a cutoff** (Week 11 lab `show_performance`). Applicants below the cutoff are predicted to be defaulters.

In [ ]:
def show_performance(title, y_actual, predicted_list):
    cm = metrics.confusion_matrix(y_actual, predicted_list, labels=[0, 1])
    tp, fn, fp, tn = cm.ravel()
    labelled = pd.DataFrame(
        [[f'{tp:,}  (TP)', f'{fn:,}  (FN)'],
         [f'{fp:,}  (FP)', f'{tn:,}  (TN)']],
        index=['Actual: Good payer (0)', 'Actual: Defaulter (1)'],
        columns=['Predicted: Good payer (0)', 'Predicted: Defaulter (1)'])
    print(f'*** {title} ***')
    print('Confusion matrix:')
    print(labelled.to_string())
    print('PCC measures:')
    print(metrics.classification_report(y_actual, predicted_list, labels=[0, 1], target_names=['Good payer (0)', 'Defaulter (1)']))

def pcc_at(cutoff):
    show_performance(f'Training Data Performance, cutoff {cutoff}', y_train, (train_score['score'] < cutoff).astype(int).tolist())
    show_performance(f'Test Data Performance, cutoff {cutoff}',     y_test,  (test_score['score']  < cutoff).astype(int).tolist())

## Part 6. Model tuning
What was compared (test Gini decides; train vs test shows overfitting):
1. **Number of variables:** the forward-selection path above shows where extra variables stop adding much.
2. **Automatic vs adjusted bins:** `sc.woebin` defaults against the monotonic `breaks_list` version.
3. **`class_weight='balanced'`** (Week 9 lab): it re-weights the bads but distorts the probabilities, so the score no longer
   lines up with the 600 = 5% PD scaling.
4. **All candidate variables** against the selected ones.

In [ ]:
def test_gini(bins_used, cols, **lr_args):
    trw = sc.woebin_ply(train[[c[:-4] for c in cols] + ['TARGET']], bins_used)
    tew = sc.woebin_ply(test[[c[:-4] for c in cols] + ['TARGET']], bins_used)
    m = linear_model.LogisticRegression(max_iter=1000, **lr_args).fit(trw[cols], trw['TARGET'])
    p_tr, p_te = m.predict_proba(trw[cols])[:, 1], m.predict_proba(tew[cols])[:, 1]
    return dict(n_vars=len(cols), gini_train=gini(trw['TARGET'], p_tr), gini_test=gini(tew['TARGET'], p_te),
                wrong_sign=int((m.coef_[0] <= 0).sum()), mean_predicted_PD=p_te.mean())

tuning = pd.DataFrame({
    'Selected variables, adjusted bins (chosen)': test_gini(bins, selected),
    'Selected variables, automatic bins':         test_gini(bins_auto, selected),
    'First 5 selected variables':                 test_gini(bins, selected[:5]),
    'First 10 selected variables':                test_gini(bins, selected[:10]),
    'All candidate variables':                    test_gini(bins, candidates),
    "Selected + class_weight='balanced'":         test_gini(bins, selected, class_weight='balanced'),
}).T
print(f"actual test bad rate: {y_test.mean():.2%}")
tuning.round(4)

**Reading the tuning table**
* **Gains flatten after about 10 variables**, so 15 is a reasonable stopping point.
* **Automatic bins score slightly higher** than the adjusted ones, because `sc.woebin` is free to follow every wiggle in
  the data. The adjusted, monotonic bins are kept because each one has a business explanation and is more stable on
  new applicants. A small Gini cost for an explainable scorecard is the trade-off to state.
* **All candidate variables** gives the highest Gini, but at least one coefficient has the wrong sign, so its points would
  contradict its own bins. It is also harder to explain and maintain.
* **`class_weight='balanced'`** barely changes the ranking (Gini), but the average predicted PD jumps far above the real
  bad rate. The scorecard scaling (600 = 5% PD) and the expected-default calculation both need true probabilities, so it is not used.

## Part 7. Cutoff, interest rate and profitability
### 7.1 Cutoff score and expected defaults (Week 11 lab)
For each cutoff: the approval rate, **expected defaults** from the score
(odds good:bad = 19 × 2^((score − 600)/20), PD = 1/(1 + odds)), and actual defaults among approved applicants.

In [ ]:
def cutoff_table(score, y, cutoffs):
    rows = []
    for cutoff in cutoffs:
        approved = score['score'] >= cutoff
        odds = 19 * 2 ** ((score.loc[approved, 'score'] - 600) / 20)
        rows.append(dict(cutoff=cutoff, approved=int(approved.sum()), approval_rate=approved.mean(),
                         expected_defaults=(1 / (1 + odds)).sum(), actual_defaults=int(y[approved].sum()),
                         bad_rate_approved=y[approved].mean()))
    return pd.DataFrame(rows)

lowest = int(min(train_score.score.min(), test_score.score.min())) // 5 * 5      # first cutoff approves everyone
cutoffs = list(range(lowest, int(train_score.score.quantile(.95)), 5))
cutoff_train = cutoff_table(train_score, y_train, cutoffs)
cutoff_train[cutoff_train.cutoff >= 540].round(4)        # below 540 almost everyone is approved

### 7.2 Profitability and interest rate
**Assumptions. Agree these as a group and state them on the slides:**

| Assumption | Value | Reason |
|---|---|---|
| Product interest rate (APR) | 15% | illustrative rate for an unsecured instalment loan |
| Cost of funds | 5% p.a. | bank's own borrowing cost |
| Loss given default (LGD) | 70% of the loan | unsecured, about 30% recovered (same as the group's first version) |
| Loan amount and term | each applicant's `AMT_CREDIT` and `TERM_MONTHS` (from `loan_info`, section 3.1) | from the data |
| TARGET = 1 | treated as a default | TARGET means payment difficulties, so this is conservative |

* **Profit on a good loan** = interest earned at the APR − interest paid at the cost of funds, over the term (monthly instalment loan).
* **Loss on a bad loan** = LGD × loan amount.
* The **profit-maximising cutoff** is chosen on the training set and then checked on the test set.
* The **break-even APR** per score band is the lowest rate at which the band covers its own losses. It is the basis for risk-based pricing.

In [ ]:
APR, COST_OF_FUNDS, LGD = 0.15, 0.05, 0.70

def total_interest(principal, months, apr):
    r = apr / 12
    payment = principal * r / (1 - (1 + r) ** -months)
    return payment * months - principal

def profit_frame(data, score):
    info = loan_info.loc[data.index]
    f = pd.DataFrame({'score': score['score'].values, 'TARGET': data['TARGET'].values,
                      'loan': info['AMT_CREDIT'].values, 'term': info['TERM_MONTHS'].fillna(info['TERM_MONTHS'].median()).round().clip(lower=6).values})
    net_interest = total_interest(f.loan, f.term, APR) - total_interest(f.loan, f.term, COST_OF_FUNDS)
    f['profit'] = np.where(f.TARGET == 0, net_interest, -LGD * f.loan)
    return f

def profit_curve(f, cutoffs):
    rows = []
    for c in cutoffs:
        a = f[f.score >= c]
        goods, bads = a[a.TARGET == 0], a[a.TARGET == 1]
        rows.append(dict(cutoff=c, approved=len(a), approval_rate=len(a) / len(f), defaults=len(bads),
                         bad_rate=len(bads) / max(len(a), 1),
                         revenue_m=goods.profit.sum() / 1e6,          # net interest earned on good loans
                         losses_m=-bads.profit.sum() / 1e6,           # LGD x loan on bad loans
                         profit_m=a.profit.sum() / 1e6))
    return pd.DataFrame(rows)

pf_train, pf_test = profit_frame(train, train_score), profit_frame(test, test_score)
curve_train = profit_curve(pf_train, cutoffs)
best_cutoff = int(curve_train.loc[curve_train.profit_m.idxmax(), 'cutoff'])
curve_test = profit_curve(pf_test, cutoffs)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curve_train.cutoff, curve_train.profit_m / 0.7, label='train (scaled to full book)')
ax.plot(curve_test.cutoff, curve_test.profit_m / 0.3, label='test (scaled to full book)')
ax.axvline(best_cutoff, ls='--', c='grey'); ax.set_xlabel('cutoff score'); ax.set_ylabel('profit (millions)')
ax.legend(); plt.tight_layout(); plt.savefig('g7_profit_curve.png', dpi=150); plt.show()

print(f'profit-maximising cutoff (train): {best_cutoff}')

# profit at every cutoff on the test set (amounts in millions of the data's currency)
profit_table = curve_test[((curve_test.cutoff % 10 == 0) & (curve_test.cutoff >= 540)) | (curve_test.cutoff == best_cutoff)]
display(profit_table.round(4))

summary_cut = pd.concat([curve_test.iloc[[0]].assign(policy='approve everyone'),
                         curve_test[curve_test.cutoff == best_cutoff].assign(policy=f'cutoff {best_cutoff}')])
summary_cut.set_index('policy').round(4)

In [ ]:
# break-even APR by score band (test set)
def breakeven_apr(g):
    lo, hi = 0.0, 2.0
    for _ in range(60):
        mid = (lo + hi) / 2
        ni = total_interest(g.loan, g.term, mid) - total_interest(g.loan, g.term, COST_OF_FUNDS)
        prof = ((1 - g.TARGET) * ni - g.TARGET * LGD * g.loan).sum()
        lo, hi = (mid, hi) if prof < 0 else (lo, mid)
    return pd.Series({'applicants': len(g), 'bad_rate': g.TARGET.mean(), 'breakeven_APR': mid})

edges = [-np.inf, best_cutoff, best_cutoff + 20, best_cutoff + 40, best_cutoff + 60, np.inf]
pf_test['band'] = pd.cut(pf_test.score, edges, right=False)
pricing = pf_test.groupby('band', observed=True)[['TARGET', 'loan', 'term']].apply(breakeven_apr)
pricing.round(4)

**Risk-based pricing tiers and origination policy.** Each approved band becomes a pricing tier. The indicative rate is
the band's break-even APR plus a margin for operating costs and profit, rounded up to the nearest 0.5%. Lower-scoring
tiers get more checks before the loan is paid out. The margin and the checks are business decisions.

In [ ]:
MARGIN = 0.03   # margin above break-even for operating costs and profit (group decision)

tiers = pricing.reset_index(drop=True)
tiers.insert(0, 'score_band', [f'below {best_cutoff}', f'{best_cutoff}-{best_cutoff + 19}', f'{best_cutoff + 20}-{best_cutoff + 39}',
                               f'{best_cutoff + 40}-{best_cutoff + 59}', f'{best_cutoff + 60} and above'])
tiers.insert(0, 'tier', ['Decline', 'Tier 4 (Higher risk)', 'Tier 3 (Standard)', 'Tier 2 (Near prime)', 'Tier 1 (Prime)'])
tiers['indicative_APR'] = np.ceil((tiers['breakeven_APR'] + MARGIN) * 200) / 200
tiers.loc[0, 'indicative_APR'] = np.nan
tiers['origination_policy'] = [
    'Decline, or refer for a co-signer or security',
    'Approve after full review: income, 3 months of bank statements, clinic treatment plan; consider a lower amount or shorter term',
    'Approve after income check (payslips) and clinic treatment plan',
    'Automatic approval with standard documents (ID, payslip, clinic quote)',
    'Automatic approval; funds paid directly to the clinic']
tiers.iloc[::-1].reset_index(drop=True).round(4)

**PCC measures at the chosen cutoff**

In [ ]:
pcc_at(best_cutoff)

**Summary measures at the chosen cutoff (test set).** As in the Week 11 lab, a good payer is the *positive* class.
* **Accuracy** = share of all applicants classified correctly; **error rate** = 1 − accuracy.
* **Sensitivity (recall)** = share of good payers who are approved.
* **Specificity** = share of defaulters who are declined.
* **Precision** = share of approved applicants who repay.
* **F1 score** = the balance of precision and recall for good payers: 2 × precision × recall / (precision + recall).

In [ ]:
pred_good = (test_score['score'].values >= best_cutoff)
actual_good = (y_test.values == 0)
TP = int((pred_good & actual_good).sum());   FN = int((~pred_good & actual_good).sum())
FP = int((pred_good & ~actual_good).sum());  TN = int((~pred_good & ~actual_good).sum())
accuracy = (TP + TN) / len(actual_good)
precision, recall = TP / (TP + FP), TP / (TP + FN)
cutoff_measures = pd.Series({
    'Accuracy': accuracy,
    'Error rate': 1 - accuracy,
    'Sensitivity / recall (good payers approved)': recall,
    'Specificity (defaulters declined)': TN / (TN + FP),
    'Precision (approved applicants who repay)': precision,
    'F1 score (good payers)': 2 * precision * recall / (precision + recall)})
cutoff_measures.map('{:.2%}'.format)

## Part 8. Scorecard walkthrough (Week 11 Ex Q1 style)
Three test applicants at the 10th, 50th and 90th percentile of score. Their points are looked up bin by bin with
`sc.scorecard_ply(..., only_total_score=False)`, added up, and compared with the cutoff.

In [ ]:
examples = []
for q in (0.10, 0.50, 0.90):
    examples.append((test_score['score'] - test_score['score'].quantile(q)).abs().idxmin())
for i in examples:
    appl = test.loc[[i], [c[:-4] for c in selected]]
    pts = sc.scorecard_ply(appl, card, only_total_score=False).transpose()
    total = int(pts.loc['score'].iloc[0])
    pts = pts.drop(index='score'); pts.index = pts.index.str.replace('_points', '')
    summary = pd.concat([appl.transpose(), pts], axis=1); summary.columns = ['App Value', 'Points']
    print(f"\nApplicant {df.loc[i, 'SK_ID_CURR']}: total {total} -> {'APPROVE' if total >= best_cutoff else 'DECLINE'} (cutoff {best_cutoff}), actual TARGET = {test.loc[i, 'TARGET']}")
    print(summary.to_string())

## Summary

In [ ]:
acc, allp = curve_test[curve_test.cutoff == best_cutoff].iloc[0], curve_test.iloc[0]
print('=' * 78); print('SCORECARD SUMMARY - G1-7 Medical Procedure Loan'); print('=' * 78)
print(f"segment {len(df):,} | train {len(train):,} | test {len(test):,} | bad rate {df['TARGET'].mean():.2%}")
print(f"screening: {len(iv_fine)} variables -> {len(screened)} fine IV >= 0.02 -> {len(coarse_kept)} coarse IV >= 0.02 "
      f"-> {len(candidates)} after WOE correlation -> {len(selected)} in the model")
print(f"performance: Gini train {perf.Gini['train']:.3f} | test {perf.Gini['test']:.3f} | KS test {perf.KS['test']:.3f}")
print(f"scaling: 600 points = 1:19 bad:good odds (PD 5%), PDO 20")
print(f"cutoff {best_cutoff}: approve {acc.approval_rate:.1%} | bad rate {acc.bad_rate:.2%} (vs {allp.bad_rate:.2%} approving all)")
print(f"test-set profit {acc.profit_m:,.0f}m vs {allp.profit_m:,.0f}m approving all ({acc.profit_m/allp.profit_m-1:+.1%})")